# Design Space Exploration with Mocasin

In the first part of this tutorial, you explored the LAKSA compiler and generated HLS code for the residual-block example.

In this part, we connect LAKSA to **Mocasin**, a rapid-prototyping framework developed at TU Dresden for mapping dataflow applications onto heterogeneous multiprocessor platforms.

We will:

1. export the LAKSA application model for Mocasin,
2. add CPU and FPGA execution profiles,
3. simulate a mapping,
4. explore mappings with a genetic algorithm, and
5. generate Pareto-optimal operating points.

## Mocasin in a nutshell

Mocasin models three main pieces of information:

- an **application graph** describing processes and communication,
- a **hardware platform** describing processors and communication resources, and
- **execution traces / profiles** describing how the application behaves on the platform.

A mapper combines these models to generate an application mapping. Mocasin can then replay the application traces with its trace-based simulator to estimate the execution time of that mapping.

Conceptually, the workflow is:

```text
Application model ─┐
Execution profiles ─┼──> Mapper ───> Mapping ───> Trace-based simulator
Platform model ─────┘
```

Mocasin provides several mapping algorithms, ranging from simple deterministic mappings to heuristic and meta-heuristic search algorithms.

## Visualize the application graph

Before exporting the application to Mocasin, inspect its dataflow topology. `ladle` can export the graph as Graphviz DOT, which Jupyter can render inline.

In [ ]:
!ladle input.mlir --dot -o application.dot

In [ ]:
from pathlib import Path
import graphviz

graphviz.Source(Path("application.dot").read_text())

## Prepare the application YAML

Mocasin reads the application topology and processor-specific execution profiles from YAML. We will generate the topology template, add profiles, and merge them into an application description.

### Generate the Mocasin application template

LAKSA exports a Mocasin-compatible template directly from the MLIR input.

In [ ]:
!ladle input.mlir --mocasin -o out_dir

The generated template (`out_dir/mocasin/template.yaml`) contains the **application topology**. The `profiles/` directory is created alongside it. The template initially uses `UNKNOWN` placeholders for process execution costs:

```yaml
execution:
  processes:
    profiles:
      boundary:
        UNKNOWN:
          cycles: 0
      main_node_0:
        UNKNOWN:
          cycles: 0
      main_node_1:
        UNKNOWN:
          cycles: 0
```

We will combine the template with processor-specific execution profiles before using it in Mocasin.

### Execution profiles

When generating HLS code, LAKSA also emits execution-profile information from its internal model:

In [ ]:
!ladle input.mlir --hls -o out_dir

The HLS flow also creates `out_dir/profiles/profiles_K26_PL_model.yaml`, with cycle estimates from LAKSA's internal FPGA model.

A complete workflow can additionally obtain FPGA profiles from the HLS synthesis reports and CPU profiles from benchmarking on the target platform.

For this notebook, the CPU benchmark and Vitis HLS synthesis are skipped. Prepared profiles are available in `residual-block-profiles`:

- `profiles_CortexA53.yaml` — CPU profiles,
- `profiles_K26_PL_hls.yaml` — FPGA profiles obtained from the HLS tool.

The HLS profile takes priority over the internal model profile for the same FPGA processor. The internal-model profile is still useful when no HLS profile is available.

In [ ]:
!cp residual-block-profiles/profiles_*.yaml out_dir/profiles/
!ls -1 out_dir/profiles

### Merge the profiles

`ladle --mocasin` generated `out_dir/mocasin/merge_profiles.sh` together with the template. Run that script to merge the template with the profile fragments in `out_dir/profiles/`. It also assigns the synthetic graph boundary processes to `CortexA53` with a one-cycle latency.

In [ ]:
!out_dir/mocasin/merge_profiles.sh

The merged application is written to `out_dir/mocasin/application.yaml`. It now contains execution costs for the available processor types and is ready to use with Mocasin.

In [ ]:
from pathlib import Path

text = Path("out_dir/mocasin/application.yaml").read_text()
print(text)

## Run Mocasin

Mocasin exposes different **tasks**, each backed by a Hydra configuration pipeline.

For this tutorial we use:

- `simulate` — generate a mapping and simulate its execution,
- `pareto_front` — explore mappings and export Pareto-optimal operating points.

### Start with the default mapper

First, use Mocasin's `default` mapper. It is intentionally simple: it maps processes to the first available processor and selects the first suitable communication primitive.

The relevant command-line parameters are:

- `simulate` — the Mocasin task,
- `graph=yaml_reader` — read the application graph from YAML,
- `trace=yaml_reader` — read the trace/profile information from YAML,
- `yaml.file=out_dir/mocasin/application.yaml` — path to the application description,
- `platform=kria_kv260` — target the Kria KV260 platform,
- `mapper=default` — use the simple default mapper.

Mocasin creates a separate Hydra run directory for each invocation. We pass an absolute path to the application YAML because Hydra changes into the run directory before running the task.

In [ ]:
!mocasin simulate \
    graph=yaml_reader \
    trace=yaml_reader \
    platform=kria_kv260 \
    mapper=default \
    yaml.file=$(pwd)/out_dir/mocasin/application.yaml \
    hydra.output_subdir=null

A successful run reports both the **simulated application time** and the **wall-clock time spent by the simulator**, for example:

```text
Total simulated time: 7.16656096 ms
Total simulation time: 0.002273... s
```

These are different quantities: the first is the estimated execution time of the mapped application; the second is how long the simulator itself took to run. The task also writes a `summary.csv` in its Hydra run directory.

### Search with the genetic mapper

The default mapper is useful as a baseline, but it does not search the design space.

Mocasin also provides a genetic mapper. The following command uses its default search parameters:

In [ ]:
!mocasin simulate \
    graph=yaml_reader \
    trace=yaml_reader \
    platform=kria_kv260 \
    mapper=genetic \
    yaml.file=$(pwd)/out_dir/mocasin/application.yaml \
    hydra.output_subdir=null

During the search, Mocasin prints one row per generation. The columns summarize the current population, including the average, standard deviation, minimum, and maximum objective values.

We can increase the search effort explicitly. Here we use a population of 20 mappings for 20 generations:

In [ ]:
!mocasin simulate \
    graph=yaml_reader \
    trace=yaml_reader \
    platform=kria_kv260 \
    mapper=genetic \
    mapper.pop_size=20 \
    mapper.num_gens=20 \
    yaml.file=$(pwd)/out_dir/mocasin/application.yaml \
    hydra.output_subdir=null

Compare the final simulated application time with the result from the default mapper. The purpose of design-space exploration is to find mappings that make better use of the heterogeneous CPU and FPGA resources rather than simply selecting the first available processor.

### Generate Pareto-optimal operating points

Instead of returning only a single mapping, the `pareto_front` task exports a set of non-dominated mappings discovered during the search.

We use the same genetic search parameters as above:

In [ ]:
!mocasin pareto_front \
    graph=yaml_reader \
    trace=yaml_reader \
    platform=kria_kv260 \
    mapper=genetic \
    mapper.pop_size=20 \
    mapper.num_gens=20 \
    yaml.file=$(pwd)/out_dir/mocasin/application.yaml \
    mapping_table=$(pwd)/mappings.csv \
    hydra.output_subdir=null

The resulting `mappings.csv` contains the Pareto-front mappings and their metadata. We can inspect it directly from the notebook:

In [ ]:
import pandas as pd

mappings = pd.read_csv("mappings.csv")
mappings